# LassoSelector

Fits a linear model with an L1 penalty and keeps the features with non-zero
coefficients.

The L1 penalty pushes most coefficients to exactly zero. What survives is the
selection. For classification it is L1 logistic regression; for regression,
Lasso.

- **Strength:** considers features together, so it can drop a feature that is
  only useful through another one.
- **Weakness:** costs a model fit. Among strongly correlated features it tends
  to keep one and zero the rest, somewhat arbitrarily.
- **Modes:** `n_features=None` keeps every non-zero coefficient, so the method
  picks its own count; `n_features=k` keeps the k largest.
- `C` sets the strength for classification: smaller `C` is sparser.

In [1]:
import numpy as np

rng = np.random.default_rng(0)
from sklearn.datasets import make_classification

from featsel.selectors import LassoSelector

X, y = make_classification(n_samples=300, n_features=200, n_informative=8,
                           n_redundant=0, shuffle=False, random_state=0)
truth = set(range(8))

## Letting the penalty choose the count

In [2]:
for C in (0.01, 0.03, 0.1, 1.0):
    chosen = LassoSelector(C=C, random_state=0).fit(X, y).get_support(indices=True)
    print(f'C={C:<5} keeps {len(chosen):>3} features, {len(truth & set(chosen))} of 8 truly informative')

C=0.01  keeps   4 features, 4 of 8 truly informative
C=0.03  keeps   4 features, 4 of 8 truly informative
C=0.1   keeps  32 features, 6 of 8 truly informative
C=1.0   keeps 128 features, 7 of 8 truly informative


## Correlated copies

Duplicate an informative feature a few times with small noise. Lasso puts
its weight mostly on one copy, and which one is close to arbitrary.

In [3]:
copies = X[:, [0]] + rng.normal(0, 0.01, size=(len(X), 4))
X_dup = np.column_stack([X, copies])            # columns 200-203 copy column 0
coef = LassoSelector(C=0.1, random_state=0).fit(X_dup, y).feature_importances_
print('weight on column 0 and its copies:', np.round(coef[[0, 200, 201, 202, 203]], 3))

weight on column 0 and its copies: [0.    0.    0.005 0.285 0.   ]
